# GraphRAG Evaluation Pipeline — ContextWeaver
## RAGAS + GraphRAG-Bench Metrics (arXiv 2506.05690)

Stack:
- Dataset: RAGAS testset ODS (Polish)
- LLM: `gpt-4o-mini` (OpenAI key from env)
- Embeddings: `text-embedding-3-small
- QA service: `POST /qa?rag_mode=HYBRID`

In [ ]:
# %pip install ragas networkx rouge-score sentence-transformers openai neo4j pandas odfpy matplotlib seaborn

In [ ]:
import os, re, ast, math, json, asyncio, warnings
from pathlib import Path
from typing import List, Dict, Optional

import httpx
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
from rouge_score import rouge_scorer
from sentence_transformers import util
from neo4j import GraphDatabase

from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness, ResponseRelevancy,
    LLMContextPrecisionWithReference, LLMContextRecall,
    FactualCorrectness, SemanticSimilarity, RougeScore,
)
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings("ignore")
load_dotenv()

## Configuration

In [ ]:
TESTSET_PATH = Path("magisterka_eval/tags/testset_clean.csv")

QA_BASE_URL = "http://localhost:8080"
QA_ENDPOINT = f"{QA_BASE_URL}/qa"

os.environ.get("OPENAI_API_KEY")

LLM_MODEL = "gpt-4o-mini"

## Loading models

In [ ]:
from openai import OpenAI

# LLM (RAGAS + LLM-based scoring)
evaluator_llm = llm_factory(LLM_MODEL, client=OpenAI())
print(f"LLM: {LLM_MODEL}")

## Loading the RAGAS dataset (ODS)

Columns from `TestsetGenerator`: `user_input`, `reference_contexts`, `reference`


In [ ]:
def parse_contexts(raw) -> List[str]:
    if isinstance(raw, list):
        return [str(x) for x in raw if x]
    if isinstance(raw, str):
        raw = raw.strip()

        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x]
        except Exception:
            pass

        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x]
        except Exception:
            pass

        return [raw] if raw else []
    return []


df_raw = pd.read_csv(TESTSET_PATH)
print(f"Loaded {len(df_raw)} rows | columns: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# Column normalization ────────────────────────────────────────────────────────
col_map = {}
for col in df_raw.columns:
    lc = col.lower().strip()
    if "user_input" in lc or lc == "question":
        col_map["user_input"] = col
    elif "reference_context" in lc:
        col_map["reference_contexts"] = col
    elif lc == "reference" or "ground_truth" in lc:
        col_map["reference"] = col

print("Column mapping:", col_map)
assert all(k in col_map for k in ["user_input", "reference_contexts", "reference"]), (
    f"Missing columns! Found: {list(df_raw.columns)}"
)

RAW_SAMPLES = []
for _, row in df_raw.iterrows():
    question = str(row[col_map["user_input"]]).strip()
    reference = str(row[col_map["reference"]]).strip()
    ref_contexts = parse_contexts(row[col_map["reference_contexts"]])
    if question and reference:
        RAW_SAMPLES.append({
            "question": question,
            "reference": reference,
            "reference_contexts": ref_contexts,
        })

print(f"Loaded {len(RAW_SAMPLES)} samples")
print(f"\nExample [0]:")
print(f"  question   : {RAW_SAMPLES[0]['question'][:80]}...")
print(f"  reference  : {RAW_SAMPLES[0]['reference'][:80]}...")
print(f"  contexts   : {len(RAW_SAMPLES[0]['reference_contexts'])} fragments")

## Calling the QA service

For each question we call the `POST /qa?rag_mode=X` endpoint in 4 modes:
`GRAPH_CHUNK`, `VANILLA`, `HYBRID`, `GRAPH`


In [ ]:
async def call_qa(question: str, rag_mode: str, timeout: float = 600.0) -> Dict:
    """Calls POST /qa?rag_mode=<mode> and returns a dict with the answer."""
    payload = {"question": question}
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(
            QA_ENDPOINT,
            json=payload,
            params={"rag_mode": rag_mode},
        )
        resp.raise_for_status()
        data = resp.json()

    answer = data.get("answer")
    retrieved = data.get("source_context")
    return {"response": answer, "retrieved_contexts": retrieved, "raw": data}


async def run_all_modes(samples: List[Dict], modes=["hybrid", "graph"]) -> Dict[str, List]:
    """Calls the service for each sample x each mode."""
    results = {mode: [] for mode in modes}
    total = len(samples) * len(modes)
    done = 0
    for s in samples:
        for mode in modes:
            try:
                out = await call_qa(s["question"], mode)
            except Exception as e:
                print(f"  [{mode}] error for '{s['question'][:40]}': {e}")
                out = {"response": "", "retrieved_contexts": [], "raw": {}}
            results[mode].append({
                **s,
                "response": out["response"],
                "retrieved_contexts": out["retrieved_contexts"],
                "rag_mode": mode,
            })
            done += 1
            if done % 5 == 0:
                print(f"  [{done}/{total}] ...")
    return results


print("Calling the QA service for all samples x 2 modes...")
qa_results = await run_all_modes(RAW_SAMPLES)
print(f"\nDone!")
for mode, rows in qa_results.items():
    non_empty = sum(1 for r in rows if r["response"])
    print(f"  {mode:8}: {non_empty}/{len(rows)} answers")

In [ ]:
for mode, rows in qa_results.items():
    print(f"\n=== {mode} ===")
    for r in rows:
        ctx = r.get("retrieved_contexts")
        print(f"  question : {r['question'][:50]}")
        print(f"  response : {repr(r['response'])}")
        print(f"  ctx type : {type(ctx)}, len: {len(ctx) if ctx else 'N/A'}")
        if ctx:
            print(f"  ctx   : {repr(ctx)[:80]}")

In [ ]:
from datetime import datetime
import csv

def save_qa_results_to_csv(qa_results: Dict[str, List], output_path: str):
    rows = []
    for mode, samples in qa_results.items():
        for r in samples:
            ctx = r.get("retrieved_contexts", "")
            if isinstance(ctx, list):
                ctx = "\n---\n".join(str(c) for c in ctx if c)

            rows.append({
                "rag_mode":           mode,
                "question":           r.get("question", ""),
                "response":           r.get("response", ""),
                "reference":          r.get("reference", ""),
                "retrieved_contexts": ctx,
                "reference_contexts": "\n---\n".join(r.get("reference_contexts", []) or []),
            })

    df = pd.DataFrame(rows)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
    )
    print(f"Saved {len(df)} rows → {output_path}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_qa_results_to_csv(
    qa_results,
    output_path=f"magisterka_eval/tags/v3/merged_v3_{timestamp}.csv"
)

## RAGAS Metrics (per mode)

In [ ]:
def to_ctx_list(ctx) -> List[str]:
    if isinstance(ctx, list):
        return ctx if ctx else ["brak kontekstu"]
    if isinstance(ctx, str) and ctx.strip():
        return [ctx]
    return ["brak kontekstu"]

ragas_metric_list = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm)
]

ragas_dfs = {}
for mode, rows in qa_results.items():
    print(f"RAGAS [{mode}]...")
    samples_eval = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["response"] or "brak odpowiedzi",
            reference=r["reference"],
            retrieved_contexts=to_ctx_list(r["retrieved_contexts"]),
            reference_contexts=to_ctx_list(r["reference_contexts"]),
        )
        for r in rows
    ]
    ds = EvaluationDataset(samples=samples_eval)
    result = evaluate(dataset=ds, metrics=ragas_metric_list)
    df_r = result.to_pandas()
    df_r["mode"] = mode
    ragas_dfs[mode] = df_r
    print(f"  {mode} OK")


first_df = next(iter(ragas_dfs.values()))
ragas_num_cols = [c for c in first_df.columns
                  if c not in ["user_input","response","retrieved_contexts",
                               "reference","reference_contexts","mode"]
                  and pd.api.types.is_numeric_dtype(first_df[c])]

summary_ragas = pd.DataFrame([
    {"mode": mode, **{col: df[col].mean() for col in ragas_num_cols}}
    for mode, df in ragas_dfs.items()
]).round(4)

print("\n" + "=" * 75)
print("RAGAS METRICS - Avg per mode")
print("=" * 75)
print(summary_ragas.to_string(index=False))

In [ ]:
import csv

all_ragas = pd.concat(ragas_dfs.values(), ignore_index=True)
all_ragas.to_csv(
    "Graph RAG Dataset/testset/contextweaver_graph_ragas_detailed.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

summary_ragas.to_csv(
    "Graph RAG Dataset/testset/contextweaver_graph_ragas_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved {len(all_ragas)} detailed rows + summary")